# GPT OSS 120B — RLVR with Tool-Integrated Reasoning

Train `openai/gpt-oss-120b` via **RLVR** with **Python tool use** using the Tinker `tinker_cookbook.rl` framework.

**Architecture:**
- **Multi-turn environment** via `MessageEnv` + `EnvFromMessageEnv`
- Model generates reasoning → calls `python` tool → gets execution output → continues → produces `\boxed{}` answer
- **Local Jupyter kernel** for code execution during rollouts (no Modal required)
- GRPO with importance sampling loss, group_size=4
- `GptOssRenderer` for Harmony format (handles `<|call|>`, `<|return|>` routing)

**Reward:**
- `+1.0` for correct `\boxed{}` answer matching ground truth
- `-1.0` for wrong answer or no `\boxed{}`
- `-0.1` for context overflow (ran out of tokens)
- `-1.0` for parse errors

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================

class RLConfig:
    model_name = "openai/gpt-oss-120b"
    lora_rank = 32
    learning_rate = 1e-5          # Lower LR for RL (vs 2e-4 for SFT)
    max_tokens = 4096             # Max tokens PER generation step (per turn)
    max_trajectory_tokens = 24576 # Total token budget for entire multi-turn episode
    batch_size = 2                # Problems per training step
    group_size = 8                # Completions per problem (GRPO)
    max_steps = 30                # Total training iterations
    temperature = 1.0             # Sampling temperature for rollouts
    loss_fn = "importance_sampling"  # Standard for GRPO
    eval_every = 30               # Evaluate every N steps
    save_every = 15               # Checkpoint every N steps
    log_path = "/kaggle/working/rl_logs"
    
    # Tool execution
    max_tool_iterations = 15      # Max tool calls per episode
    python_timeout = 30.0         # Timeout for each Python execution (seconds)
    
    # Dataset split
    dataset_path = "/kaggle/input/datasets/nahidhossainredom/rlvr-dataset/rlvr_dataset.csv"
    train_last_n = 10             # Literal last N rows become training data
    eval_size = 1                 # Seeded-random sample from remaining rows
    split_seed = 42               # Seed for validation row sampling
    
    # Optional: warm-start from SFT checkpoint
    load_checkpoint_path = None  # e.g. "tinker://SESSION_ID:train:0/sampler_weights/NAME"

print("RLVR + Tool Use Config:")
for k, v in vars(RLConfig).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")


In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

!pip install -q tinker tinker-cookbook

In [ ]:
# ============================================================
# Cell 3: Imports & API Key (UNCHANGED)
# ============================================================

import os, re, math, logging, asyncio, threading, queue, contextlib, io, traceback
from functools import partial
from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import Literal, cast

import pandas as pd
import chz
import tinker

from tinker_cookbook import renderers, model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.rl.types import (
    RLDataset, RLDatasetBuilder, EnvGroupBuilder, Env,
    Trajectory, Metrics, StepResult, Action, ActionExtra,
)
from tinker_cookbook.rl.message_env import (
    MessageEnv, MessageStepResult, EnvFromMessageEnv,
)
from tinker_cookbook.rl import train
from tinker_cookbook.completers import StopCondition
from tinker_cookbook.renderers.base import Message, ToolSpec

# Try to import the cookbook's math grading utilities
try:
    from tinker_cookbook.recipes.math_rl.math_grading import (
        extract_boxed,
        grade_answer,
        run_with_timeout_signal,
    )
    HAS_MATH_GRADING = True
    print("\u2713 Loaded tinker_cookbook math grading (SymPy-based)")
except ImportError:
    HAS_MATH_GRADING = False
    print("\u26a0 Math grading not available, using string matching fallback")

# ---- SET YOUR TINKER API KEY HERE ----
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gpt-oss-rlvr-tool")

print(f"Tinker SDK version: {tinker.__version__}")
print(f"API key set: {'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'}")

In [ ]:
# ============================================================
# Cell 4: Load & Inspect Dataset
# ============================================================

df = pd.read_csv(RLConfig.dataset_path)

print(f"Dataset loaded: {len(df)} problems")
print(f"Columns: {list(df.columns)}")
print(f"\nAnswer types:")
print(f"  Numeric (int-like): {df['answer'].apply(lambda x: str(x).lstrip('-').isdigit()).sum()}")
print(f"  Other:             {(~df['answer'].apply(lambda x: str(x).lstrip('-').isdigit())).sum()}")

# Preview
print(f"\n{'='*60}")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    # print(f"\n[Problem {row['id']}]")
    print(f"  Q: {str(row['problem'])[:150]}...")
    print(f"  A: {row['answer']}")

In [ ]:
# ============================================================
# Cell 5: Python Execution Tool — Persistent Worker Executor
# ============================================================
# Uses one worker process per environment so Python state persists across
# tool calls, but timeouts can still be enforced by terminating the worker.

import json
import multiprocessing as mp


class CodeExecutor:
    """Persistent Python executor with hard timeout recovery.

    Each environment gets its own worker process. The worker keeps a shared
    globals namespace for the current episode, so variables persist across
    tool calls. If execution times out or the worker crashes, the process is
    terminated and recreated so runaway code cannot accumulate in the main
    notebook process.
    """

    _PREAMBLE = (
        "import math\n"
        "import numpy as np\n"
        "import sympy as sp\n"
        "from sympy import *\n"
        "import itertools\n"
        "import collections\n"
        "import warnings; warnings.filterwarnings('ignore')\n"
    )

    _MAX_OUTPUT_CHARS = 3000

    def __init__(self, timeout: float = 30.0):
        self._timeout = timeout
        # Using spawn to avoid fork-related memory inflation and deadlocks
        # in the asyncio-heavy Jupyter kernel.
        self._ctx = mp.get_context("spawn")
        self._lock = threading.Lock()
        self._task_queue = None
        self._result_queue = None
        self._process = None
        self._start_worker()

    @staticmethod
    def _worker_main(task_queue, result_queue, preamble: str, max_output_chars: int):
        globals_dict = {"__builtins__": __builtins__}
        exec(preamble, globals_dict)

        while True:
            command = task_queue.get()
            if command is None:
                break

            action = command.get("action")

            if action == "reset":
                globals_dict = {"__builtins__": __builtins__}
                exec(preamble, globals_dict)
                result_queue.put({"status": "reset", "output": "[RESET] OK"})
                continue

            if action != "execute":
                result_queue.put({
                    "status": "executor_error",
                    "output": f"[ERROR] Unknown executor action: {action}",
                })
                continue

            code = command.get("code", "")
            buf = io.StringIO()

            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, globals_dict)
                stdout = buf.getvalue().strip()
                output = stdout or "[No output. Use print() to see results.]"
            except Exception as exc:
                stdout = buf.getvalue().strip()
                tb_lines = traceback.format_exception(type(exc), exc, exc.__traceback__)
                clean_lines = []
                skip = True
                for line in tb_lines:
                    if "<string>" in line or "exec(" not in line:
                        skip = False
                    if not skip:
                        clean_lines.append(line)
                if not clean_lines:
                    clean_lines = [f"{type(exc).__name__}: {exc}\n"]
                error = "".join(clean_lines).strip()
                output = f"{stdout}\n{error}" if stdout else error

            if len(output) > max_output_chars:
                output = output[:max_output_chars] + f"\n... [truncated, {len(output)} total chars]"

            result_queue.put({"status": "ok", "output": output})

    def _start_worker(self):
        self._task_queue = self._ctx.Queue()
        self._result_queue = self._ctx.Queue()
        self._process = self._ctx.Process(
            target=self._worker_main,
            args=(self._task_queue, self._result_queue, self._PREAMBLE, self._MAX_OUTPUT_CHARS),
            daemon=True,
        )
        self._process.start()

    def _terminate_worker(self):
        if self._task_queue is not None:
            with contextlib.suppress(Exception):
                self._task_queue.put_nowait(None)

        if self._process is not None:
            with contextlib.suppress(Exception):
                if self._process.is_alive():
                    self._process.terminate()
                self._process.join(timeout=1.0)
                if self._process.is_alive():
                    self._process.kill()
                    self._process.join(timeout=1.0)

        for queue_obj in (self._task_queue, self._result_queue):
            if queue_obj is not None:
                with contextlib.suppress(Exception):
                    queue_obj.close()
                with contextlib.suppress(Exception):
                    queue_obj.join_thread()

        self._task_queue = None
        self._result_queue = None
        self._process = None

    def _restart_worker(self):
        self._terminate_worker()
        self._start_worker()

    def _drain_results(self):
        if self._result_queue is None:
            return
        while True:
            try:
                self._result_queue.get_nowait()
            except queue.Empty:
                break

    def _round_trip(self, payload: dict, timeout: float) -> dict:
        with self._lock:
            if self._process is None or not self._process.is_alive():
                self._restart_worker()

            self._drain_results()
            self._task_queue.put(payload)

            try:
                return self._result_queue.get(timeout=timeout)
            except queue.Empty:
                self._restart_worker()
                return {
                    "status": "timeout",
                    "output": f"[TIMEOUT] Execution exceeded {timeout:.0f}s",
                }
            except Exception as exc:
                self._restart_worker()
                return {
                    "status": "executor_error",
                    "output": f"[ERROR] ExecutorFailure: {type(exc).__name__}: {exc}",
                }

    def execute(self, code: str, timeout: float | None = None) -> dict:
        effective_timeout = timeout or self._timeout
        return self._round_trip({"action": "execute", "code": code}, effective_timeout)

    def reset(self):
        response = self._round_trip({"action": "reset"}, self._timeout)
        if response.get("status") not in {"ok", "reset"}:
            self._restart_worker()

    def close(self):
        with self._lock:
            self._terminate_worker()


# Quick test
print("Testing CodeExecutor...")
_test = CodeExecutor(timeout=5.0)

result = _test.execute("print(2 + 2)")
assert result["status"] == "ok" and "4" in result["output"], f"Basic exec failed: {result}"
print(f"  2+2 = {result['output'].strip()}")

_test.execute("x = 42")
result = _test.execute("print(x * 2)")
assert result["status"] == "ok" and "84" in result["output"], f"State persistence failed: {result}"
print(f"  x*2 = {result['output'].strip()}")

result = _test.execute("print(isprime(17))")
assert result["status"] == "ok" and "True" in result["output"], f"SymPy test failed: {result}"
print(f"  isprime(17) = {result['output'].strip()}")

result = _test.execute("1/0")
assert result["status"] == "ok" and "ZeroDivision" in result["output"], f"Error handling failed: {result}"
print(f"  1/0 → {result['output'].strip()[:50]}")

result = _test.execute("while True: pass", timeout=2.0)
assert result["status"] == "timeout", f"Timeout test failed: {result}"
print(f"  infinite loop → {result['output'].strip()}")

_test.reset()
result = _test.execute("try:\n    print(x)\nexcept NameError:\n    print('RESET OK')")
assert result["status"] == "ok" and "RESET OK" in result["output"], f"Reset test failed: {result}"
print(f"  after reset → {result['output'].strip()}")

_test.close()
print("\u2713 CodeExecutor works — hard timeouts and per-episode state persistence")

In [ ]:
# ============================================================
# Cell 6: Answer Grading Utilities (UNCHANGED)
# ============================================================

def extract_boxed_answer(text: str) -> str | None:
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    while i < len(text) and depth:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[idx + len(key):i - 1].strip() if depth == 0 else None


def safe_grade_answer(given: str, ground_truth: str, timeout: float = 2.0) -> bool:
    """Grade an answer using SymPy if available, else string matching."""
    if HAS_MATH_GRADING:
        try:
            result = run_with_timeout_signal(
                grade_answer,
                args=(given, ground_truth),
                timeout_seconds=int(math.ceil(timeout)),
            )
            if result is not None:
                return result
        except Exception:
            pass
    
    # Fallback: normalize and compare strings
    def normalize(s: str) -> str:
        s = s.strip().replace(" ", "").replace(",", "")
        if s.endswith(".0"):
            s = s[:-2]
        return s.lower()
    
    return normalize(given) == normalize(ground_truth)


# Quick test
print("Answer grading tests:")
print(f"  extract_boxed_answer('... \\\\boxed{{42}}') = {extract_boxed_answer('The answer is \\\\boxed{42}')}")
print(f"  extract_boxed_answer('no box') = {extract_boxed_answer('no box')}")
print(f"  safe_grade_answer('42', '42') = {safe_grade_answer('42', '42')}")
print(f"  safe_grade_answer('43', '42') = {safe_grade_answer('43', '42')}")
print("\u2713 Grading OK")


In [ ]:
# ============================================================
# Cell 7: Tool-Use Math Environment (MessageEnv subclass)
# ============================================================

from tinker_cookbook.utils import logtree

PYTHON_TOOL_SPEC: ToolSpec = {
    "name": "python",
    "description": (
        "Execute Python code. The environment is a stateful notebook with "
        "math, numpy (as np), and sympy (as sp) pre-imported. Always use "
        "print() to display results. Code persists between calls."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "Python code to execute"
            }
        },
        "required": ["code"]
    }
}

SYSTEM_PROMPT = (
    "You are a mathematical problem solver with access to a Python tool.\n\n"
    "# Strategy:\n"
    "1. Think through the problem step by step\n"
    "2. Use the python tool to verify computations and explore\n"
    "3. When confident in your answer, put it inside \\boxed{}\n\n"
    "# Tool protocol:\n"
    "- Preferred format: send a tool call to recipient `python` on the commentary channel\n"
    "- Put the tool payload in JSON with a `code` field, e.g. {\\\"code\\\": \\\"print(2+2)\\\"}\n"
    "- Legacy `functions.python` calls are still accepted\n\n"
    "# Rules:\n"
    "- Use print() in your code to see results\n"
    "- You can call the tool multiple times\n"
    "- Put your FINAL answer in \\boxed{} format (e.g., \\boxed{42})\n"
    "- Do NOT just guess - verify with computation when possible"
)


class ToolUseMathEnv(MessageEnv):
    """Multi-turn math environment with robust tool-call recovery."""

    def __init__(
        self,
        problem: str,
        answer: str,
        code_executor: CodeExecutor,
        renderer_name: str,
        model_name: str,
        python_timeout: float = 30.0,
        max_iterations: int = 15,
    ):
        self.problem = problem
        self.answer = str(answer).strip()
        self.executor = code_executor
        self.renderer_name = renderer_name
        self.model_name = model_name
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.iteration = 0
        self.format_failures = 0
        self.consecutive_no_progress = 0
        self._conversation: list[Message] = []

    async def initial_observation(self) -> list[Message]:
        tokenizer = get_tokenizer(self.model_name)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)

        prefix_messages = renderer.create_conversation_prefix_with_tools(
            tools=[PYTHON_TOOL_SPEC],
            system_prompt=SYSTEM_PROMPT,
        )

        user_msg: Message = {
            "role": "user",
            "content": (
                self.problem +
                "\n\nPlease reason step by step, use the python tool to verify "
                "your computations, and put your final answer within \\boxed{}."
            ),
        }
        self._conversation = prefix_messages + [user_msg]
        return list(self._conversation)

    def _message_content_text(self, message: Message) -> str:
        content = message.get("content", "")
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            chunks = []
            for item in content:
                if isinstance(item, dict):
                    chunks.append(item.get("text", ""))
                    chunks.append(item.get("thinking", ""))
                else:
                    chunks.append(str(item))
            return " ".join(chunk for chunk in chunks if chunk)
        return str(content)

    def _message_blob(self, message: Message) -> str:
        parts = [self._message_content_text(message)]
        parts.append(str(message.get("recipient", "")))
        parts.append(str(message.get("channel", "")))

        for tool_call in message.get("tool_calls", []):
            function = getattr(tool_call, "function", None)
            if function is not None:
                parts.append(str(getattr(function, "name", "")))
                parts.append(str(getattr(function, "arguments", "")))
            else:
                parts.append(str(tool_call))

        for raw_call in message.get("unparsed_tool_calls", []):
            parts.append(str(getattr(raw_call, "raw_text", raw_call)))

        return "\n".join(part for part in parts if part)

    def _normalize_tool_name(self, name: str | None) -> str | None:
        if name in {"python", "functions.python"}:
            return "python"
        return None

    def _extract_code_from_payload(self, payload, *, allow_direct_code: bool) -> str | None:
        if payload is None:
            return None

        if isinstance(payload, dict):
            code = payload.get("code")
            return code.strip() if isinstance(code, str) and code.strip() else None

        text = str(payload).strip()
        if not text:
            return None

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = None

        if isinstance(parsed, dict):
            code = parsed.get("code")
            if isinstance(code, str) and code.strip():
                return code.strip()

        if "<|message|>" in text:
            inner = text.split("<|message|>", 1)[1]
            inner = re.split(r"<\|(?:call|end)\|>", inner, maxsplit=1)[0].strip()
            return self._extract_code_from_payload(inner, allow_direct_code=True)

        fenced = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
        if fenced:
            code = fenced.group(1).strip()
            if code:
                return code

        json_like = re.search(r'\{.*?"code"\s*:\s*.*?\}', text, re.S)
        if json_like:
            try:
                parsed = json.loads(json_like.group(0))
            except json.JSONDecodeError:
                parsed = None
            if isinstance(parsed, dict):
                code = parsed.get("code")
                if isinstance(code, str) and code.strip():
                    return code.strip()

        if allow_direct_code and not text.startswith("{"):
            return text

        return None

    def _looks_like_tool_attempt(self, message: Message) -> bool:
        if self._normalize_tool_name(message.get("recipient")):
            return True
        if message.get("tool_calls") or message.get("unparsed_tool_calls"):
            return True

        blob = self._message_blob(message)
        return any(marker in blob for marker in (
            "to=python",
            "to=functions.python",
            '"code"',
            "<|call|>",
        ))

    def _extract_tool_intent(self, message: Message) -> tuple[dict | None, bool]:
        blob = self._message_blob(message)

        for tool_call in message.get("tool_calls", []):
            function = getattr(tool_call, "function", None)
            if function is None:
                continue
            name = getattr(function, "name", None)
            if self._normalize_tool_name(name) != "python":
                continue
            code = self._extract_code_from_payload(getattr(function, "arguments", None), allow_direct_code=True)
            if code:
                canonical = "to=functions.python" not in blob and self._normalize_tool_name(message.get("recipient")) == "python"
                return ({
                    "tool_name": "python",
                    "code": code,
                    "source": "tool_calls",
                    "canonical": canonical,
                }, False)
            return (None, True)

        recipient = message.get("recipient")
        if self._normalize_tool_name(recipient) == "python":
            code = self._extract_code_from_payload(self._message_content_text(message), allow_direct_code=True)
            if code:
                return ({
                    "tool_name": "python",
                    "code": code,
                    "source": "recipient",
                    "canonical": recipient == "python",
                }, False)
            return (None, True)

        for raw_call in message.get("unparsed_tool_calls", []):
            raw_text = getattr(raw_call, "raw_text", raw_call)
            code = self._extract_code_from_payload(raw_text, allow_direct_code=False)
            if code:
                canonical = "to=python" in str(raw_text) and "functions.python" not in str(raw_text)
                return ({
                    "tool_name": "python",
                    "code": code,
                    "source": "unparsed_tool_calls",
                    "canonical": canonical,
                }, False)

        if self._looks_like_tool_attempt(message):
            code = self._extract_code_from_payload(blob, allow_direct_code=False)
            if code:
                canonical = "to=python" in blob and "functions.python" not in blob
                return ({
                    "tool_name": "python",
                    "code": code,
                    "source": "raw_fallback",
                    "canonical": canonical,
                }, False)
            return (None, True)

        return (None, False)

    def _check_for_boxed_answer(self, message: Message) -> str | None:
        return extract_boxed_answer(self._message_content_text(message))

    def _make_tool_response(self, output: str, channel: str) -> Message:
        return {
            "role": "tool",
            "name": "python",
            "recipient": "assistant",
            "channel": channel,
            "content": output,
        }

    async def step(self, message: Message) -> MessageStepResult:
        self.iteration += 1
        self._conversation.append(message)

        predicted = self._check_for_boxed_answer(message)
        if predicted is not None:
            correct = safe_grade_answer(predicted, self.answer)
            reward = 1.0 if correct else -1.0

            with logtree.scope_header("Answer Check"):
                logtree.table_from_dict({
                    "predicted": predicted,
                    "ground_truth": self.answer,
                    "correct": correct,
                    "reward": reward,
                    "iterations": self.iteration,
                }, caption="Final answer")

            return MessageStepResult(
                reward=reward,
                episode_done=True,
                next_messages=[],
                metrics={"correct": float(correct), "iterations": self.iteration},
            )

        if message.get("channel") == "final":
            return MessageStepResult(
                reward=-1.0,
                episode_done=True,
                next_messages=[],
                metrics={"invalid_final": 1.0, "iterations": self.iteration},
            )

        intent, malformed = self._extract_tool_intent(message)
        if intent is not None:
            self.format_failures = 0
            self.consecutive_no_progress = 0

            try:
                exec_result = await asyncio.to_thread(
                    self.executor.execute,
                    intent["code"],
                    self.python_timeout,
                )
            except Exception as exc:
                exec_result = {
                    "status": "executor_error",
                    "output": f"[ERROR] ExecutorFailure: {type(exc).__name__}: {exc}",
                }

            output = exec_result["output"]
            channel = message.get("channel") or "commentary"
            self._conversation.append(self._make_tool_response(output, channel=channel))

            metrics = {
                "tool_call_ok": 1.0,
                "iterations": self.iteration,
                "canonical_python_recipient": float(intent["canonical"]),
                "fallback_parse_used": float(intent["source"] not in {"tool_calls", "recipient"}),
            }

            if exec_result["status"] == "timeout":
                metrics["tool_timeout"] = 1.0
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics=metrics,
                )

            if exec_result["status"] != "ok":
                metrics["tool_executor_error"] = 1.0
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics=metrics,
                )

            if self.iteration >= self.max_iterations:
                metrics["max_iterations_hit"] = 1.0
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics=metrics,
                )

            return MessageStepResult(
                reward=0.0,
                episode_done=False,
                next_messages=list(self._conversation),
                metrics=metrics,
            )

        if malformed:
            self.consecutive_no_progress = 0
            self.format_failures += 1
            error_msg = (
                "[FORMAT ERROR] Tool calls must target recipient `python` on the commentary channel "
                "and provide JSON like {\"code\": \"print(2+2)\"}. Legacy `functions.python` is accepted."
            )
            self._conversation.append(self._make_tool_response(error_msg, channel="commentary"))

            metrics = {
                "tool_format_error": 1.0,
                "iterations": self.iteration,
                "fallback_parse_used": 1.0,
            }

            if self.format_failures >= 2 or self.iteration >= self.max_iterations:
                if self.iteration >= self.max_iterations:
                    metrics["max_iterations_hit"] = 1.0
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics=metrics,
                )

            return MessageStepResult(
                reward=-0.25,
                episode_done=False,
                next_messages=list(self._conversation),
                metrics=metrics,
            )

        self.consecutive_no_progress += 1

        if self.iteration >= self.max_iterations:
            return MessageStepResult(
                reward=-1.0,
                episode_done=True,
                next_messages=[],
                metrics={"correct": 0.0, "no_answer": 1.0, "iterations": self.iteration},
            )

        if self.consecutive_no_progress >= 2:
            return MessageStepResult(
                reward=-1.0,
                episode_done=True,
                next_messages=[],
                metrics={"stalled": 1.0, "iterations": self.iteration},
            )

        return MessageStepResult(
            reward=0.0,
            episode_done=False,
            next_messages=list(self._conversation),
            metrics={"no_tool_no_answer": 1.0, "iterations": self.iteration},
        )


print("\u2713 ToolUseMathEnv defined")
print("  Canonical tool route: recipient `python` on commentary channel")
print("  Fallback parser accepts legacy `functions.python` and malformed Harmony payloads")
print(f"  Max {RLConfig.max_tool_iterations} tool iterations, {RLConfig.python_timeout}s timeout per exec")
print("  Invalid final / malformed tool / stalled trajectories terminate early")

In [ ]:
# ============================================================
# Cell 8: EnvGroupBuilder with Tool Execution
# ============================================================

@dataclass
class ToolMathGroupBuilder(EnvGroupBuilder):
    """Builds a group of ToolUseMathEnv wrapped in EnvFromMessageEnv."""

    problem: str = ""
    answer: str = ""
    num_envs: int = 1
    renderer_name: str = ""
    model_name: str = ""
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    _executors: list[CodeExecutor] = field(default_factory=list, init=False, repr=False)

    async def make_envs(self) -> Sequence[Env]:
        """Create num_envs ToolUseMathEnv instances, each with its own executor."""
        tokenizer = get_tokenizer(self.model_name)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)

        envs = []
        self._executors = []

        for _ in range(self.num_envs):
            executor = CodeExecutor(timeout=self.python_timeout)
            self._executors.append(executor)

            msg_env = ToolUseMathEnv(
                problem=self.problem,
                answer=self.answer,
                code_executor=executor,
                renderer_name=self.renderer_name,
                model_name=self.model_name,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
            )

            env = EnvFromMessageEnv(
                renderer=renderer,
                message_env=msg_env,
                failed_parse_reward=-1.0,
                terminate_on_parse_error=True,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
                context_overflow_reward=-0.1,
            )
            envs.append(env)

        return envs

    async def compute_group_rewards(
        self, trajectory_group: list[Trajectory], env_group: Sequence[Env]
    ) -> list[tuple[float, Metrics]]:
        """No additional group-level rewards (all rewards come from step)."""
        return [(0.0, {}) for _ in trajectory_group]

    async def cleanup(self) -> None:
        """Close executor workers allocated for this env group."""
        for executor in self._executors:
            executor.close()
        self._executors.clear()

    def logging_tags(self) -> list[str]:
        return ["aimo_tool"]


print("\u2713 ToolMathGroupBuilder defined")
print(f"  Each group: {RLConfig.group_size} envs, each with its own persistent executor")
print(f"  Cleanup closes worker processes after each group")
print(f"  Max trajectory tokens: {RLConfig.max_trajectory_tokens}")
print(f"  Max generation tokens per turn: {RLConfig.max_tokens}")

In [ ]:
# ============================================================
# Cell 9: Custom RLDataset & RLDatasetBuilder
# ============================================================

class AIMOToolDataset(RLDataset):
    """RL dataset wrapping our CSV with tool-use environment builders."""
    
    def __init__(
        self,
        problems: list[dict],
        batch_size: int,
        group_size: int,
        renderer_name: str,
        model_name: str,
        python_timeout: float = 30.0,
        max_iterations: int = 15,
        max_trajectory_tokens: int = 24576,
        max_generation_tokens: int = 4096,
    ):
        self.problems = problems
        self.batch_size = batch_size
        self.group_size = group_size
        self.renderer_name = renderer_name
        self.model_name = model_name
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.max_trajectory_tokens = max_trajectory_tokens
        self.max_generation_tokens = max_generation_tokens
    
    def get_batch(self, index: int) -> Sequence[EnvGroupBuilder]:
        batch_start = (index * self.batch_size) % len(self.problems)
        indices = []
        for i in range(self.batch_size):
            indices.append((batch_start + i) % len(self.problems))
        
        builders = []
        for i in indices:
            p = self.problems[i]
            builder = ToolMathGroupBuilder(
                problem=p["problem"],
                answer=str(p["answer"]),
                num_envs=self.group_size,
                renderer_name=self.renderer_name,
                model_name=self.model_name,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
            )
            builders.append(builder)
        return builders
    
    def __len__(self) -> int:
        return max(1000, math.ceil(len(self.problems) / self.batch_size))


@chz.chz
class AIMOToolDatasetBuilder(RLDatasetBuilder):
    """Builds train/eval datasets from our CSV with a tiny fixed split."""
    
    dataset_path: str
    batch_size: int
    group_size: int
    model_name_for_tokenizer: str
    renderer_name: str
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    train_last_n: int = 10
    eval_size: int = 1
    split_seed: int = 42
    
    async def __call__(self) -> tuple[AIMOToolDataset, AIMOToolDataset | None]:
        import random
        
        df = pd.read_csv(self.dataset_path)
        all_problems = df.to_dict('records')
        
        min_required = self.train_last_n + self.eval_size
        if len(all_problems) < min_required:
            raise ValueError(
                f"Dataset must have at least {min_required} rows for train_last_n={self.train_last_n} and eval_size={self.eval_size}"
            )

        train_problems = all_problems[-self.train_last_n:]
        eval_pool = all_problems[:-self.train_last_n]

        rng = random.Random(self.split_seed)
        eval_indices = rng.sample(range(len(eval_pool)), k=self.eval_size)
        eval_problems = [eval_pool[i] for i in eval_indices]

        logger.info(
            "Dataset split: %s train (last %s rows), %s eval (seed=%s sample from %s remaining rows)",
            len(train_problems),
            self.train_last_n,
            len(eval_problems),
            self.split_seed,
            len(eval_pool),
        )
        train_dataset = AIMOToolDataset(
            problems=train_problems,
            batch_size=self.batch_size,
            group_size=self.group_size,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        eval_dataset = AIMOToolDataset(
            problems=eval_problems,
            batch_size=len(eval_problems),
            group_size=1,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        return train_dataset, eval_dataset


print("\u2713 AIMOToolDataset and AIMOToolDatasetBuilder defined")
print(f"  {len(df)} total problems")
print(f"  Train split: literal last {RLConfig.train_last_n} rows")
print(f"  Eval split: seeded sample of {RLConfig.eval_size} row from remaining {len(df) - RLConfig.train_last_n}")
print(f"  Each step: {RLConfig.batch_size} problems x {RLConfig.group_size} completions = {RLConfig.batch_size * RLConfig.group_size} rollouts")
print(f"  Each rollout: up to {RLConfig.max_tool_iterations} tool calls")
print("  Executor: persistent per-env worker process")


In [ ]:
# ============================================================
# Cell 10: Preflight Checks
# ============================================================

preflight_renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)
print(f"Preflight renderer: {preflight_renderer_name}")

_preflight_executor = CodeExecutor(timeout=2.0)
result = _preflight_executor.execute("print(21 * 2)")
assert result["status"] == "ok" and result["output"].strip() == "42", result

_preflight_executor.execute("value = 7")
result = _preflight_executor.execute("print(value ** 2)")
assert result["status"] == "ok" and result["output"].strip() == "49", result

result = _preflight_executor.execute("while True: pass", timeout=1.0)
assert result["status"] == "timeout", result

_preflight_executor.reset()
result = _preflight_executor.execute("try:\n    print(value)\nexcept NameError:\n    print('RESET OK')")
assert result["status"] == "ok" and "RESET OK" in result["output"], result

_preflight_env = ToolUseMathEnv(
    problem="What is 2 + 2?",
    answer="4",
    code_executor=CodeExecutor(timeout=2.0),
    renderer_name=preflight_renderer_name,
    model_name=RLConfig.model_name,
    python_timeout=2.0,
    max_iterations=RLConfig.max_tool_iterations,
)

intent, malformed = _preflight_env._extract_tool_intent({
    "role": "assistant",
    "recipient": "python",
    "channel": "commentary",
    "content": '{"code": "print(2 + 2)"}',
})
assert intent is not None and intent["code"] == "print(2 + 2)" and not malformed

intent, malformed = _preflight_env._extract_tool_intent({
    "role": "assistant",
    "recipient": "functions.python",
    "channel": "commentary",
    "content": '{"code": "print(5)"}',
})
assert intent is not None and intent["code"] == "print(5)" and not malformed

intent, malformed = _preflight_env._extract_tool_intent({
    "role": "assistant",
    "content": '<|start|>assistant to=functions.python<|channel|>commentary <|constrain|>json<|message|>{"code": "print(9)"}<|call|>',
})
assert intent is not None and intent["code"] == "print(9)" and not malformed

result = await _preflight_env.step({
    "role": "assistant",
    "recipient": "python",
    "channel": "commentary",
    "content": '{"code": "print(2 + 2)"}',
})
assert result.reward == 0.0 and not result.episode_done and result.metrics["tool_call_ok"] == 1.0

result = await _preflight_env.step({
    "role": "assistant",
    "channel": "final",
    "content": '',
})
assert result.episode_done and result.metrics["invalid_final"] == 1.0

result = await _preflight_env.step({
    "role": "assistant",
    "recipient": "python",
    "channel": "commentary",
    "content": '{"code": ',
})
assert result.metrics["tool_format_error"] == 1.0 and not result.episode_done

result = await _preflight_env.step({
    "role": "assistant",
    "content": 'Still thinking without using a tool yet.',
})
assert not result.episode_done

result = await _preflight_env.step({
    "role": "assistant",
    "content": 'Still thinking without using a tool yet.',
})
assert result.episode_done and result.metrics["stalled"] == 1.0

_preflight_env.executor.close()

preflight_dataset_builder = AIMOToolDatasetBuilder(
    dataset_path=RLConfig.dataset_path,
    batch_size=RLConfig.batch_size,
    group_size=RLConfig.group_size,
    model_name_for_tokenizer=RLConfig.model_name,
    renderer_name=preflight_renderer_name,
    python_timeout=RLConfig.python_timeout,
    max_iterations=RLConfig.max_tool_iterations,
    max_trajectory_tokens=RLConfig.max_trajectory_tokens,
    max_generation_tokens=RLConfig.max_tokens,
    train_last_n=RLConfig.train_last_n,
    eval_size=RLConfig.eval_size,
    split_seed=RLConfig.split_seed,
)
train_dataset, eval_dataset = await preflight_dataset_builder()

assert len(train_dataset.problems) == RLConfig.train_last_n
assert len(eval_dataset.problems) == RLConfig.eval_size
expected_train = df.tail(RLConfig.train_last_n).to_dict('records')
assert train_dataset.problems == expected_train
remaining_pool = df.iloc[:-RLConfig.train_last_n].to_dict('records')
assert eval_dataset.problems[0] in remaining_pool
assert eval_dataset.problems[0] not in train_dataset.problems

_preflight_executor.close()
print("\u2713 Preflight checks passed")
print(f"  Train rows: {len(train_dataset.problems)} | Eval rows: {len(eval_dataset.problems)}")
print("  Parser, executor, invalid final, stalled turn, and split checks are all green")


In [ ]:
# ============================================================
# Cell 11: Configure & Launch RL Training
# ============================================================

renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)
print(f"Using renderer: {renderer_name}")

dataset_builder = AIMOToolDatasetBuilder(
    dataset_path=RLConfig.dataset_path,
    batch_size=RLConfig.batch_size,
    group_size=RLConfig.group_size,
    model_name_for_tokenizer=RLConfig.model_name,
    renderer_name=renderer_name,
    python_timeout=RLConfig.python_timeout,
    max_iterations=RLConfig.max_tool_iterations,
    max_trajectory_tokens=RLConfig.max_trajectory_tokens,
    max_generation_tokens=RLConfig.max_tokens,
    train_last_n=RLConfig.train_last_n,
    eval_size=RLConfig.eval_size,
    split_seed=RLConfig.split_seed,
)

config = train.Config(
    model_name=RLConfig.model_name,
    learning_rate=RLConfig.learning_rate,
    dataset_builder=dataset_builder,
    max_tokens=RLConfig.max_tokens,
    log_path=RLConfig.log_path,
    lora_rank=RLConfig.lora_rank,
    loss_fn=RLConfig.loss_fn,
    temperature=RLConfig.temperature,
    eval_every=RLConfig.eval_every,
    save_every=RLConfig.save_every,
    max_steps=RLConfig.max_steps,
    renderer_name=renderer_name,
    load_checkpoint_path=RLConfig.load_checkpoint_path,
    rollout_error_tolerance=True,
    num_groups_to_log=2,
)

print("\u2554" + "\u2550"*60 + "\u2557")
print("\u2551  GPT OSS 120B RLVR + Tool-Integrated Reasoning")
print("\u2560" + "\u2550"*60 + "\u2563")
print(f"\u2551  Model:           {config.model_name}")
print(f"\u2551  LoRA rank:       {config.lora_rank}")
print(f"\u2551  Learning rate:   {config.learning_rate:.1e}")
print(f"\u2551  Max tokens/turn: {RLConfig.max_tokens}")
print(f"\u2551  Max trajectory:  {RLConfig.max_trajectory_tokens} tokens")
print(f"\u2551  Batch size:      {RLConfig.batch_size} problems")
print(f"\u2551  Group size:      {RLConfig.group_size} completions/problem")
print(f"\u2551  Rollouts/step:   {RLConfig.batch_size * RLConfig.group_size}")
print(f"\u2551  Tool iters:      Up to {RLConfig.max_tool_iterations} per episode")
print(f"\u2551  Python timeout:  {RLConfig.python_timeout}s")
print(f"\u2551  Max steps:       {config.max_steps}")
print(f"\u2551  Train rows:      last {RLConfig.train_last_n} dataset rows")
print(f"\u2551  Eval rows:       {RLConfig.eval_size} seeded sample from remaining rows")
print(f"\u2551  Renderer:        {config.renderer_name}")
print(f"\u2551  Error tolerance: {config.rollout_error_tolerance}")
print(f"\u2551  Checkpoint:      {config.load_checkpoint_path or 'None (base model)'}")
print("\u255a" + "\u2550"*60 + "\u255d")
print("\n\u23f3 Launching RLVR training with tool use...\n")

await train.main(config)


In [ ]:
# ============================================================
# Cell 11: Download Trained Weights
# ============================================================

import requests
import json
from pathlib import Path

def find_final_checkpoint(log_path: str) -> str | None:
    """Find the final checkpoint path from the training logs."""
    checkpoints_file = Path(log_path) / "checkpoints.jsonl"
    if not checkpoints_file.exists():
        print(f"No checkpoints file found at {checkpoints_file}")
        return None
    
    last_checkpoint = None
    with open(checkpoints_file) as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                if 'sampler_path' in data:
                    last_checkpoint = data['sampler_path']
            except json.JSONDecodeError:
                continue
    return last_checkpoint


def download_weights(sampler_path: str, output_dir: str = "gpt_oss_120b_rlvr_tool_weights"):
    """Download the trained LoRA weights from Tinker."""
    os.makedirs(output_dir, exist_ok=True)
    
    service_client = tinker.ServiceClient()
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print(f"Downloading checkpoint from: {sampler_path}")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get('content-length', 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    output_file = os.path.join(output_dir, "lora_checkpoint.tar")
    downloaded = 0
    with open(output_file, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  \u2713 Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# Find and download the final checkpoint
sampler_path = find_final_checkpoint(RLConfig.log_path)
if sampler_path:
    print(f"Found checkpoint: {sampler_path}")
    download_weights(sampler_path)
else:
    print("No checkpoint found. Training may not have completed.")